# Dashcam Vehicle Proximity Analysis

This notebook processes a `.mp4` dashcam video and tracks **cars**, **buses**, and **trucks**.

For every frame:
- We run object detection + tracking.
- We keep persistent track IDs for each vehicle.
- We draw bounding boxes around target vehicles.
- We estimate proximity (`close` vs `medium` vs `far`) using bounding box area in pixels.

For every **15-second interval**:
- Each tracked vehicle is counted only once per interval (no frame-by-frame overcounting).
- We aggregate into these columns:
  - `car_close`, `car_close_stopped`, `car_medium`, `car_medium_stopped`, `car_far`, `car_far_stopped`
  - `bus_close`, `bus_close_stopped`, `bus_medium`, `bus_medium_stopped`, `bus_far`, `bus_far_stopped`
  - `truck_close`, `truck_close_stopped`, `truck_medium`, `truck_medium_stopped`, `truck_far`, `truck_far_stopped`
- We append one row to a CSV file, with the interval timestamp.

---

## How proximity is estimated

This notebook uses two configurable thresholds:

- If `bbox_area >= CLOSE_AREA_THRESHOLD_PX2` -> vehicle is `close`
- Else if `bbox_area >= MEDIUM_AREA_THRESHOLD_PX2` -> vehicle is `medium`
- Else -> vehicle is `far`

Tune these based on camera position, field of view, and desired sensitivity.

## How stopped vehicles are estimated

A tracked vehicle is marked as **stopped** only if BOTH are true for at least `STOPPED_SECONDS` seconds:

1. The vehicle box is stable:
   - Center displacement <= `STATIONARY_CENTER_DELTA_PX`
   - Area change ratio <= `STATIONARY_AREA_CHANGE_RATIO`
2. The surrounding scene is static:
   - Global frame-to-frame scene motion <= `SCENE_STATIC_MAX_MOTION_PX`

If the surroundings are moving (typical while dashcam car is moving), vehicles are classified as **moving**.

> Note: Bounding-box area and image-plane motion are proxies and not true metric distance/speed measurements.

## 1) Install and import required libraries

If needed, uncomment and run the install cell below.

- `ultralytics` provides YOLO object detection models.
- `opencv-python` handles video input/output and drawing boxes.
- `pandas` helps create/export tabular CSV output.
- `tqdm` shows processing progress and ETA.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO


## 2) Configuration

Set your input video path and adjust key parameters:

- `VIDEO_PATH`: dashcam `.mp4` file
- `START_DATETIME`: ISO 8601 with UTC offset (e.g. `2026-01-19 07:50:22+00:00` or `2026-01-19T07:50:22Z`)
- `SAMPLE_EVERY_SECONDS`: aggregation interval (n seconds)
- `CLOSE_AREA_THRESHOLD_PX2`: area threshold for `close`
- `MEDIUM_AREA_THRESHOLD_PX2`: area threshold for `medium`
- `STOPPED_SECONDS`: stationary time required to classify as stopped
- `STATIONARY_CENTER_DELTA_PX`: max center movement to still be considered stationary
- `STATIONARY_AREA_CHANGE_RATIO`: max bbox area ratio change for stationary status
- `SCENE_STATIC_MAX_MOTION_PX`: max global scene motion to treat surroundings as static
- `CONFIDENCE_THRESHOLD`: minimum detector confidence
- `MODEL_WEIGHTS`: YOLO model checkpoint
- `TRACKER_CONFIG`: YOLO tracker config (ByteTrack by default)

In [ ]:
def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "Code" else cwd

ROOT = _repo_root()
VIDEO_PATH = str(ROOT / "Artifacts" / "dashcam_input.mp4")
START_DATETIME = "2026-01-01T00:00:00+00:00"

OUTPUT_VIDEO_PATH = str(ROOT / "Artifacts" / "dashcam_annotated.mp4")
OUTPUT_CSV_PATH = str(ROOT / "Artifacts" / "vehicle_proximity_counts.csv")

SAMPLE_EVERY_SECONDS = 30
CLOSE_AREA_THRESHOLD_PX2 = 30000
MEDIUM_AREA_THRESHOLD_PX2 = 14000
STOPPED_SECONDS = 4.0
STATIONARY_CENTER_DELTA_PX = 12.0
STATIONARY_AREA_CHANGE_RATIO = 0.08
SCENE_STATIC_MAX_MOTION_PX = 0.8
CONFIDENCE_THRESHOLD = 0.70
MODEL_WEIGHTS = str(ROOT / "Code" / "yolov8n.pt")
TRACKER_CONFIG = "bytetrack.yaml"

TARGET_CLASSES = {
    "car": "car",
    "bus": "bus",
    "truck": "truck",
}



## 3) Helper functions

- `classify_proximity`: labels each detection as `close`, `medium`, or `far` from bbox area.
- `init_interval_counter`: creates a zero-initialized counter for one interval.
- `build_interval_counts_from_tracks`: converts unique track records to interval counts.
- `interval_row_template`: standard output row format.

In [ ]:
def classify_proximity(bbox_area: float, close_threshold: float, medium_threshold: float) -> str:
    """Return close/medium/far from bbox area thresholds."""
    if close_threshold < medium_threshold:
        raise ValueError("CLOSE_AREA_THRESHOLD_PX2 must be >= MEDIUM_AREA_THRESHOLD_PX2")

    if bbox_area >= close_threshold:
        return "close"
    if bbox_area >= medium_threshold:
        return "medium"
    return "far"


def init_interval_counter() -> dict:
    """Initialize count dictionary for a single interval."""
    return {
        "car_close": 0,
        "car_close_stopped": 0,
        "car_medium": 0,
        "car_medium_stopped": 0,
        "car_far": 0,
        "car_far_stopped": 0,
        "bus_close": 0,
        "bus_close_stopped": 0,
        "bus_medium": 0,
        "bus_medium_stopped": 0,
        "bus_far": 0,
        "bus_far_stopped": 0,
        "truck_close": 0,
        "truck_close_stopped": 0,
        "truck_medium": 0,
        "truck_medium_stopped": 0,
        "truck_far": 0,
        "truck_far_stopped": 0,
    }


def build_interval_counts_from_tracks(unique_tracks: dict) -> dict:
    """Build one interval count object from unique track records.

    unique_tracks format:
      key -> {"class_name": str, "proximity": str, "stopped": bool}
    """
    counts = init_interval_counter()

    for record in unique_tracks.values():
        cls_name = record["class_name"]
        proximity = record["proximity"]
        stopped = record["stopped"]

        base_key = f"{cls_name}_{proximity}"
        stopped_key = f"{base_key}_stopped"

        if stopped:
            if stopped_key in counts:
                counts[stopped_key] += 1
        else:
            if base_key in counts:
                counts[base_key] += 1

    return counts


def interval_row_template(timestamp: str, counts: dict) -> dict:
    """Create a standardized row for CSV output."""
    return {
        "timestamp": timestamp,
        "car_close": counts["car_close"],
        "car_close_stopped": counts["car_close_stopped"],
        "car_medium": counts["car_medium"],
        "car_medium_stopped": counts["car_medium_stopped"],
        "car_far": counts["car_far"],
        "car_far_stopped": counts["car_far_stopped"],
        "bus_close": counts["bus_close"],
        "bus_close_stopped": counts["bus_close_stopped"],
        "bus_medium": counts["bus_medium"],
        "bus_medium_stopped": counts["bus_medium_stopped"],
        "bus_far": counts["bus_far"],
        "bus_far_stopped": counts["bus_far_stopped"],
        "truck_close": counts["truck_close"],
        "truck_close_stopped": counts["truck_close_stopped"],
        "truck_medium": counts["truck_medium"],
        "truck_medium_stopped": counts["truck_medium_stopped"],
        "truck_far": counts["truck_far"],
        "truck_far_stopped": counts["truck_far_stopped"],
    }


## 4) Run tracking, annotate video, and aggregate every 10 seconds

This cell:

1. Loads YOLO.
2. Reads the input video frame-by-frame.
3. Detects + tracks vehicles (`car`, `bus`, `truck`) using track IDs.
4. Classifies each tracked vehicle as `close`, `medium`, or `far`.
5. Estimates global scene motion from frame-to-frame optical flow.
6. Classifies each tracked vehicle as `stopped` only when bbox is stable AND scene is static for at least 4 seconds.
7. Counts each unique track at most once per interval.
8. Writes annotated video and CSV output.

In [ ]:
from datetime import datetime, timedelta, timezone

video_path = Path(VIDEO_PATH)
if not video_path.exists():
    raise FileNotFoundError(f"Video file not found: {video_path}")

_start_ts = pd.to_datetime(START_DATETIME, utc=True, errors="coerce")
if pd.isna(_start_ts):
    raise ValueError(
        "Invalid START_DATETIME. Use ISO 8601 with UTC offset, e.g. '2026-01-19 07:50:22+00:00' or '2026-01-19T07:50:22Z'."
    )
start_datetime = _start_ts.to_pydatetime()

if CLOSE_AREA_THRESHOLD_PX2 < MEDIUM_AREA_THRESHOLD_PX2:
    raise ValueError("CLOSE_AREA_THRESHOLD_PX2 must be >= MEDIUM_AREA_THRESHOLD_PX2")

print("Loading model...")
model = YOLO(MODEL_WEIGHTS)

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError("Could not open video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_seconds = total_frames / fps if fps > 0 else 0

print(f"FPS: {fps:.2f}")
print(f"Resolution: {frame_width}x{frame_height}")
print(f"Total frames: {total_frames}")
print(f"Duration (s): {duration_seconds:.2f}")

Path(OUTPUT_VIDEO_PATH).parent.mkdir(parents=True, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

rows = []
current_interval_index = 0

interval_unique_tracks = {}

track_motion_state = {}

model_names = model.names
prev_gray = None

def estimate_scene_motion_px(prev_img_gray: np.ndarray, curr_img_gray: np.ndarray) -> float:
    """Estimate global scene motion in pixels using sparse optical flow.

    Lower value means scene appears static; higher means camera/environment motion.
    """
    if prev_img_gray is None or curr_img_gray is None:
        return float("inf")

    points_prev = cv2.goodFeaturesToTrack(
        prev_img_gray,
        maxCorners=260,
        qualityLevel=0.01,
        minDistance=8,
        blockSize=7,
    )

    if points_prev is None or len(points_prev) < 20:
        return float("inf")

    points_curr, status, _ = cv2.calcOpticalFlowPyrLK(prev_img_gray, curr_img_gray, points_prev, None)
    if points_curr is None or status is None:
        return float("inf")

    valid_prev = points_prev[status.flatten() == 1]
    valid_curr = points_curr[status.flatten() == 1]
    if len(valid_prev) < 20:
        return float("inf")

    displacement = np.linalg.norm(valid_curr - valid_prev, axis=1)
    return float(np.median(displacement))

def format_wall_clock(ts: datetime) -> str:
    """Format as ISO 8601 UTC with offset (same convention as START_DATETIME)."""
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=timezone.utc)
    else:
        ts = ts.astimezone(timezone.utc)
    return ts.isoformat(sep=" ", timespec="seconds")

print(f"Start datetime: {format_wall_clock(start_datetime)}")

def flush_interval(interval_idx: int) -> None:
    """Convert one interval's unique track map into one CSV row."""
    interval_start_seconds = interval_idx * SAMPLE_EVERY_SECONDS
    interval_datetime = start_datetime + timedelta(seconds=interval_start_seconds)
    unique_tracks = interval_unique_tracks.get(interval_idx, {})
    counts = build_interval_counts_from_tracks(unique_tracks)
    rows.append(interval_row_template(format_wall_clock(interval_datetime), counts))

frame_idx = 0
video_pbar = tqdm(
    total=total_frames if total_frames > 0 else None,
    desc="Processing video",
    unit="frame",
    dynamic_ncols=True,
)
while True:
    ret, frame = cap.read()
    if not ret:
        break

    timestamp_seconds = int(frame_idx / fps) if fps > 0 else 0
    timestamp_float = frame_idx / fps if fps > 0 else 0.0
    current_datetime = start_datetime + timedelta(seconds=timestamp_float)
    interval_index = timestamp_seconds // SAMPLE_EVERY_SECONDS

    curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    scene_motion_px = estimate_scene_motion_px(prev_gray, curr_gray) if prev_gray is not None else float("inf")
    scene_is_static = scene_motion_px <= SCENE_STATIC_MAX_MOTION_PX
    prev_gray = curr_gray

    while interval_index > current_interval_index:
        flush_interval(current_interval_index)
        current_interval_index += 1

    if interval_index not in interval_unique_tracks:
        interval_unique_tracks[interval_index] = {}

    results = model.track(
        frame,
        conf=CONFIDENCE_THRESHOLD,
        persist=True,
        tracker=TRACKER_CONFIG,
        verbose=False,
    )
    result = results[0]

    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            if box.id is None:
                continue

            track_id = int(box.id[0].item())
            cls_id = int(box.cls[0].item())
            cls_name = model_names.get(cls_id, str(cls_id)) if isinstance(model_names, dict) else model_names[cls_id]

            if cls_name not in TARGET_CLASSES:
                continue

            conf = float(box.conf[0].item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

            width = max(0, x2 - x1)
            height = max(0, y2 - y1)
            bbox_area = width * height
            proximity = classify_proximity(
                bbox_area,
                close_threshold=CLOSE_AREA_THRESHOLD_PX2,
                medium_threshold=MEDIUM_AREA_THRESHOLD_PX2,
            )

            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0
            track_key = (cls_name, track_id)

            state = track_motion_state.get(track_key)
            if state is None:
                state = {
                    "last_center": (cx, cy),
                    "last_area": float(bbox_area),
                    "last_timestamp": timestamp_float,
                    "stationary_time": 0.0,
                    "stopped": False,
                }
            else:
                prev_cx, prev_cy = state["last_center"]
                prev_area = max(1.0, state["last_area"])
                dt = max(0.0, timestamp_float - state["last_timestamp"])

                center_displacement = float(np.hypot(cx - prev_cx, cy - prev_cy))
                area_change_ratio = abs(float(bbox_area) - prev_area) / prev_area

                vehicle_box_is_stable = (
                    center_displacement <= STATIONARY_CENTER_DELTA_PX
                    and area_change_ratio <= STATIONARY_AREA_CHANGE_RATIO
                )

                if vehicle_box_is_stable and scene_is_static:
                    state["stationary_time"] += dt
                else:
                    state["stationary_time"] = 0.0
                    state["stopped"] = False

                if state["stationary_time"] >= STOPPED_SECONDS:
                    state["stopped"] = True

                state["last_center"] = (cx, cy)
                state["last_area"] = float(bbox_area)
                state["last_timestamp"] = timestamp_float

            track_motion_state[track_key] = state

            interval_records = interval_unique_tracks[interval_index]
            if track_key not in interval_records:
                interval_records[track_key] = {
                    "class_name": cls_name,
                    "proximity": proximity,
                    "stopped": state["stopped"],
                }
            else:
                priority = {"far": 0, "medium": 1, "close": 2}
                existing = interval_records[track_key]["proximity"]
                if priority[proximity] > priority[existing]:
                    interval_records[track_key]["proximity"] = proximity
                interval_records[track_key]["stopped"] = interval_records[track_key]["stopped"] or state["stopped"]

            proximity_color = {
                "close": (0, 255, 0),
                "medium": (0, 255, 255),
                "far": (0, 165, 255),
            }
            color = proximity_color[proximity]
            status_label = "stopped" if state["stopped"] else "moving"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = (
                f"{cls_name} #{track_id} | {proximity} | {status_label} | "
                f"scene_motion={scene_motion_px:.2f}px | conf={conf:.2f}"
            )
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2,
                cv2.LINE_AA,
            )

    overlay_text = (
        f"time={format_wall_clock(current_datetime)} | interval={current_interval_index} | "
        f"scene={'static' if scene_is_static else 'moving'} ({scene_motion_px:.2f}px)"
    )
    cv2.putText(
        frame,
        overlay_text,
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )

    writer.write(frame)
    frame_idx += 1
    if fps > 0:
        video_pbar.set_postfix(
            video=f"{timestamp_float:.1f}s / {duration_seconds:.1f}s",
            refresh=False,
        )
    video_pbar.update(1)

video_pbar.close()

if frame_idx > 0:
    final_timestamp_seconds = int((frame_idx - 1) / fps) if fps > 0 else 0
    final_interval_index = final_timestamp_seconds // SAMPLE_EVERY_SECONDS
    while current_interval_index <= final_interval_index:
        flush_interval(current_interval_index)
        current_interval_index += 1

cap.release()
writer.release()

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print("Processing complete.")
print(f"Annotated video saved to: {Path(OUTPUT_VIDEO_PATH)}")
print(f"CSV saved to: {Path(OUTPUT_CSV_PATH)}")
print(f"Rows written: {len(df)}")



## 5) Inspect results

Use this cell to preview output table and verify the expected columns for datetime-based timestamps, proximity, and stopped counts.

In [ ]:
if Path(OUTPUT_CSV_PATH).exists():
    result_df = pd.read_csv(OUTPUT_CSV_PATH)
    display(result_df.head(10))
    print("Columns:", list(result_df.columns))
else:
    print("CSV file not found yet. Run the processing cell first.")


## 6) Tuning guidance

- Increase `CLOSE_AREA_THRESHOLD_PX2` if too many vehicles are marked as `close`.
- Increase `MEDIUM_AREA_THRESHOLD_PX2` if too many vehicles are marked as `medium` (must stay <= close threshold).
- Decrease `STATIONARY_CENTER_DELTA_PX` to make box-position stability stricter.
- Decrease `STATIONARY_AREA_CHANGE_RATIO` to make box-size stability stricter.
- Decrease `SCENE_STATIC_MAX_MOTION_PX` to require a more static global scene.
- Increase `STOPPED_SECONDS` if you only want longer stops labeled as stopped.
- Increase `CONFIDENCE_THRESHOLD` to reduce false positives.